# 01 — Setup & Data Audit
**Reddit Opinion & Community Research — Pipeline step 1 of 11**




In [ ]:
import sys, importlib

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        get_ipython().system(f"pip install -q {pkg}")

for pkg in ["pandas", "pyarrow", "numpy"]:
    ensure(pkg)

print("Packages OK.")


Packages OK.


In [ ]:
import os
import json
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = "/content/drive/MyDrive/PhD/reddit_opinion_research"
else:
    # Not running in Colab (e.g. Kaggle or local). On Kaggle, upload the same three
    # files as a private Dataset and set BASE_DIR to "/kaggle/input/<your-dataset-slug>".
    BASE_DIR = os.environ.get("REDDIT_RESEARCH_BASE_DIR", "./reddit_opinion_research")

# Handle either casing of the subfolder name defensively.
_candidates = [os.path.join(BASE_DIR, "Processed"), os.path.join(BASE_DIR, "processed")]
PROCESSED_DIR = next((p for p in _candidates if os.path.isdir(p)), _candidates[0])

print("Running in Colab:", IN_COLAB)
print("BASE_DIR:      ", BASE_DIR)
print("PROCESSED_DIR: ", PROCESSED_DIR)
assert os.path.isdir(PROCESSED_DIR), f"Processed dir not found at {PROCESSED_DIR} -- check the path/case."


Mounted at /content/drive
Running in Colab: True
BASE_DIR:       /content/drive/MyDrive/PhD/reddit_opinion_research
PROCESSED_DIR:  /content/drive/MyDrive/PhD/reddit_opinion_research/processed


In [ ]:
FILES = {
    "text_corpus": os.path.join(PROCESSED_DIR, "text_corpus.parquet"),
    "graph_nodes": os.path.join(PROCESSED_DIR, "graph_nodes.parquet"),
    "graph_edges": os.path.join(PROCESSED_DIR, "graph_edges.parquet"),
}

for name, path in FILES.items():
    status = "FOUND" if os.path.isfile(path) else "MISSING"
    print(f"{name:14s} [{status}] -> {path}")


text_corpus    [FOUND] -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed/text_corpus.parquet
graph_nodes    [FOUND] -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed/graph_nodes.parquet
graph_edges    [FOUND] -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed/graph_edges.parquet


## Load & profile each table

Row counts come from Parquet metadata first (no full read needed), then each table is fully loaded for the dtype/null/sample checks below.

In [ ]:
def profile_parquet(path, name, sample_rows=5):
    print("=" * 100)
    print(f"TABLE: {name}")
    print("=" * 100)

    pf = pq.ParquetFile(path)
    print(f"Rows (from metadata): {pf.metadata.num_rows:,}  |  Row groups: {pf.metadata.num_row_groups}")

    df = pd.read_parquet(path)
    print(f"Loaded shape: {df.shape}")
    print(f"In-memory size: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

    print("\nColumns & dtypes:")
    print(df.dtypes)

    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print("\nColumns with nulls:")
    print(nulls if len(nulls) else "  (none)")

    print("\nSample rows:")
    display(df.sample(min(sample_rows, len(df)), random_state=42))

    return df


In [ ]:
text_corpus = profile_parquet(FILES["text_corpus"], "text_corpus")

TABLE: text_corpus
Rows (from metadata): 210,994  |  Row groups: 1
Loaded shape: (210994, 7)
In-memory size: 121.4 MB

Columns & dtypes:
doc_id             object
text               object
created_utc         int64
author_hash        object
thread_id          object
sentiment_label    object
topic_id           object
dtype: object

Columns with nulls:
sentiment_label    210994
topic_id           210994
dtype: int64

Sample rows:


,doc_id,text,created_utc,author_hash,thread_id,sentiment_label,topic_id
108926,comment_oc1em9o,Zlibrary,1774281821,ea2b814a50d01c1f,1s1k9ov,None,None
173928,comment_on3ri37,Liar. You could reverse back to the same spot.,1779387867,6e28013091f61c87,1tjsxt5,None,None
10326,comment_nx2yce4,"I dont think it’s an Algerian problem, people ...",1767277065,94ec480cee4c52a1,1pxas2e,None,None
40349,comment_o0mraaq,"Hi u/Comfortable_Cell_341, your post has been ...",1768895867,ac31a11084d41dec,1qhv3ys,None,None
48086,comment_o1olv03,Women need to be careful and treat foreigners ...,1769372164,2478c06f06e3a55d,1qmdd46,None,None


In [ ]:
graph_nodes = profile_parquet(FILES["graph_nodes"], "graph_nodes")

TABLE: graph_nodes
Rows (from metadata): 21,104  |  Row groups: 1
Loaded shape: (21104, 4)
In-memory size: 2.6 MB

Columns & dtypes:
author_hash      object
degree            int64
community_id     object
avg_sentiment    object
dtype: object

Columns with nulls:
community_id     21104
avg_sentiment    21104
dtype: int64

Sample rows:


,author_hash,degree,community_id,avg_sentiment
2608,f343eecca5c98628,3,None,None
20507,ec86fe00bb40cfc8,1,None,None
8284,73572a1ba3a24117,12,None,None
1978,de0e7652442d8fa4,80,None,None
3350,0951d318f0310739,1,None,None


In [ ]:
graph_edges = profile_parquet(FILES["graph_edges"], "graph_edges")

TABLE: graph_edges
Rows (from metadata): 147,964  |  Row groups: 1
Loaded shape: (147964, 3)
In-memory size: 20.4 MB

Columns & dtypes:
source_author    object
target_author    object
weight            int64
dtype: object

Columns with nulls:
  (none)

Sample rows:


,source_author,target_author,weight
141098,f617075102d81d02,0ce3e4c323cae58f,1
39590,49264e111da070f5,651b89f23dd33eba,1
80620,9210357af9253aa3,326940eacfc07856,1
1450,02f13a28d607a5d5,0e97f8cb3cda5b90,1
54837,60c20595b5eaf056,6f523edc415c6325,1


## Sanity-check against the known r/Algeria corpus statistics

You previously reported the full corpus as **183,811 documents** (9,697 posts + 174,114 comments), **180,812 monolingual** / **2,999 code-switched**, median length **18 tokens**.



In [ ]:
REFERENCE = {
    "total_documents": 183_811,
    "posts": 9_697,
    "comments": 174_114,
    "monolingual": 180_812,
    "code_switched": 2_999,
    "median_length_tokens": 18,
}

n = len(text_corpus)
print(f"Rows in text_corpus.parquet: {n:,}")
print(f"Reference full-corpus total: {REFERENCE['total_documents']:,}")
if n == REFERENCE["total_documents"]:
    print("-> Matches the full corpus exactly.")
elif abs(n - REFERENCE["total_documents"]) / REFERENCE["total_documents"] < 0.02:
    print("-> Close to the full corpus (within 2%) -- probably the full corpus with minor cleaning differences.")
else:
    print("-> Does NOT match the full corpus. Likely a subset (e.g. the ~1k annotated sample) -- confirm which one this is.")

def find_cols(df, keywords):
    return [c for c in df.columns if any(k in c.lower() for k in keywords)]

candidate_type_cols = find_cols(text_corpus, ["type", "kind", "is_post", "is_comment"])
candidate_lang_cols = find_cols(text_corpus, ["lang", "switch", "monolingual"])
candidate_len_cols  = find_cols(text_corpus, ["len", "token", "length"])

print("\nCandidate document-type column(s): ", candidate_type_cols or "none detected -- inspect manually")
print("Candidate language/code-switch column(s):", candidate_lang_cols or "none detected -- inspect manually")
print("Candidate length column(s):        ", candidate_len_cols or "none detected -- inspect manually")

for col in candidate_type_cols:
    print(f"\nValue counts for '{col}':")
    print(text_corpus[col].value_counts())

for col in candidate_lang_cols:
    print(f"\nValue counts for '{col}':")
    print(text_corpus[col].value_counts())

for col in candidate_len_cols:
    print(f"\n'{col}' summary stats:")
    print(text_corpus[col].describe())


Rows in text_corpus.parquet: 210,994
Reference full-corpus total: 183,811
-> Does NOT match the full corpus. Likely a subset (e.g. the ~1k annotated sample) -- confirm which one this is.

Candidate document-type column(s):  none detected -- inspect manually
Candidate language/code-switch column(s): none detected -- inspect manually
Candidate length column(s):         none detected -- inspect manually


## Node types (`graph_nodes.parquet`)

This is what HOD-Mamba's node-type embedding (user / comment / post / subreddit) will be built from.

In [ ]:
candidate_node_type_cols = find_cols(graph_nodes, ["type", "kind", "role"])
print("Candidate node-type column(s):", candidate_node_type_cols or "none detected -- inspect graph_nodes.columns manually")

for col in candidate_node_type_cols:
    print(f"\nValue counts for '{col}':")
    print(graph_nodes[col].value_counts())


Candidate node-type column(s): none detected -- inspect graph_nodes.columns manually


## Edge structure (`graph_edges.parquet`)

This determines how sub-threads (HOD-Mamba's hyperedges) get reconstructed, and which relation types (reply / co-participation / cross-post / subreddit-membership) actually exist in your data.


In [ ]:
candidate_edge_type_cols = find_cols(graph_edges, ["type", "relation", "kind"])
candidate_src_cols  = find_cols(graph_edges, ["parent", "reply", "source", "src", "from"])
candidate_dst_cols  = find_cols(graph_edges, ["child", "target", "dst", "to"])
candidate_time_cols = find_cols(graph_edges, ["time", "date", "created", "timestamp"])

print("Candidate edge-type column(s):  ", candidate_edge_type_cols or "none detected")
print("Candidate source/parent column(s):", candidate_src_cols or "none detected")
print("Candidate target/child column(s): ", candidate_dst_cols or "none detected")
print("Candidate timestamp column(s):   ", candidate_time_cols or "none detected")

for col in candidate_edge_type_cols:
    print(f"\nValue counts for '{col}':")
    print(graph_edges[col].value_counts())


Candidate edge-type column(s):   none detected
Candidate source/parent column(s): ['source_author']
Candidate target/child column(s):  ['target_author']
Candidate timestamp column(s):    none detected


In [ ]:
for col in candidate_time_cols:
    try:
        ts = pd.to_datetime(graph_edges[col])
        span_days = (ts.max() - ts.min()).days
        print(f"'{col}' range: {ts.min()} -> {ts.max()}  ({span_days} days)")
    except Exception as e:
        print(f"Could not parse '{col}' as a datetime: {e}")


## Thread-size / degree distribution

This sets the max sequence length HOD-Mamba's fine-grained scan needs to handle per hyperedge -- long-tailed threads will need truncation or bucketing.

In [ ]:
if candidate_src_cols and candidate_dst_cols:
    src_col, dst_col = candidate_src_cols[0], candidate_dst_cols[0]
    degree = pd.concat([graph_edges[src_col], graph_edges[dst_col]]).value_counts()
    print(f"Using source column '{src_col}' and target column '{dst_col}'")
    print(degree.describe())
    print("\n90th / 95th / 99th percentile degree:")
    print(degree.quantile([0.90, 0.95, 0.99]))
else:
    print("Could not auto-detect source/target columns -- inspect graph_edges.columns manually and rerun this cell with the right names.")


Using source column 'source_author' and target column 'target_author'
count    21104.000000
mean        14.022365
std         42.599192
min          1.000000
25%          1.000000
50%          3.000000
75%         10.000000
max       1549.000000
Name: count, dtype: float64

90th / 95th / 99th percentile degree:
0.90     31.0
0.95     59.0
0.99    180.0
Name: count, dtype: float64


## Write the data contract

Downstream notebooks load this file instead of re-guessing column names.

In [ ]:
data_contract = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "base_dir": BASE_DIR,
    "processed_dir": PROCESSED_DIR,
    "reference_corpus_stats": REFERENCE,
    "tables": {
        "text_corpus": {
            "path": FILES["text_corpus"],
            "n_rows": int(len(text_corpus)),
            "columns": {c: str(t) for c, t in text_corpus.dtypes.items()},
        },
        "graph_nodes": {
            "path": FILES["graph_nodes"],
            "n_rows": int(len(graph_nodes)),
            "columns": {c: str(t) for c, t in graph_nodes.dtypes.items()},
        },
        "graph_edges": {
            "path": FILES["graph_edges"],
            "n_rows": int(len(graph_edges)),
            "columns": {c: str(t) for c, t in graph_edges.dtypes.items()},
        },
    },
    "detected_columns": {
        "text_corpus_type_cols": candidate_type_cols,
        "text_corpus_lang_cols": candidate_lang_cols,
        "text_corpus_len_cols": candidate_len_cols,
        "node_type_cols": candidate_node_type_cols,
        "edge_type_cols": candidate_edge_type_cols,
        "edge_source_cols": candidate_src_cols,
        "edge_target_cols": candidate_dst_cols,
        "edge_time_cols": candidate_time_cols,
    },
}

contract_path = os.path.join(PROCESSED_DIR, "data_contract.json")
with open(contract_path, "w", encoding="utf-8") as f:
    json.dump(data_contract, f, indent=2, ensure_ascii=False, default=str)

print(f"Data contract written to: {contract_path}\n")
print(json.dumps(data_contract["detected_columns"], indent=2, ensure_ascii=False))


Data contract written to: /content/drive/MyDrive/PhD/reddit_opinion_research/processed/data_contract.json

{
  "text_corpus_type_cols": [],
  "text_corpus_lang_cols": [],
  "text_corpus_len_cols": [],
  "node_type_cols": [],
  "edge_type_cols": [],
  "edge_source_cols": [
    "source_author"
  ],
  "edge_target_cols": [
    "target_author"
  ],
  "edge_time_cols": []
}
